# Score d'accessibilité à l'échelle du bâtiment à Avignon

Projet personnel de Victorine Maximilienne Djetcheu Toko.

Je mesure ici ce qu'on peut atteindre à pied dans Avignon, et en combien de minutes, puis je le montre sur des cartes claires. Je m'appuie sur une méthode classique d'analyse d'accessibilité urbaine, que j'ai mise en œuvre sur ma ville.

Tout le code de cette version et tous les choix sont les miens. J'ai construit ce travail sur Avignon pour bien comprendre chaque étape et pouvoir le réutiliser sur n'importe quel territoire.

Voici mon plan. Je délimite Avignon et je récupère ses bâtiments. Je lis les arrêts du réseau de bus Orizo. Je construis le réseau piéton avec pandana. Je mesure le temps de marche jusqu'aux aménités du quotidien, pharmacie, école, commerce, santé et parc. J'attribue un score d'accessibilité à chaque bâtiment. Et je termine par des bandes d'atteignabilité à pied depuis le centre.

## 1. Les bibliothèques

J'importe mes outils. pandana me sert à calculer des distances sur le réseau piéton très vite. osmnx me donne le réseau de rues et les données d'OpenStreetMap. geopandas gère mes couches géographiques.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import pandana
import networkx as nx
import matplotlib.pyplot as plt
import time

ox.settings.overpass_rate_limit = False
ox.settings.requests_timeout = 300
ox.settings.timeout = 300

# Les serveurs publics d'OpenStreetMap sont souvent satures.
# Cette fonction essaie plusieurs serveurs, l'un apres l'autre,
# jusqu'a ce que l'un reponde. Je lance, et ca se debrouille.
SERVEURS = [
    "https://overpass.private.coffee/api",
    "https://overpass.kumi.systems/api",
    "https://overpass-api.de/api",
    "https://maps.mail.ru/osm/tools/overpass/api",
]
def tenace(fonction, essais=3):
    derniere = None
    for tour in range(essais):
        for url in SERVEURS:
            ox.settings.overpass_url = url
            nom = url.split("//")[1].split("/")[0]
            try:
                print("essai sur", nom, "...")
                return fonction()
            except Exception as e:
                derniere = e
                print("  echec,", type(e).__name__)
                time.sleep(3)
    raise derniere


## 2. Mes réglages

Je pose mes choix au même endroit, pour les retrouver facilement.

Je travaille en Lambert 93, la projection officielle en France métropolitaine, pour mesurer des distances en mètres qui soient justes. Je prends une vitesse de marche de 4,8 kilomètres par heure, soit 80 mètres par minute, c'est une marche normale. Mes seuils de lecture sont 5, 10 et 15 minutes. Et je garde un mode test, pour travailler vite sur un petit carré au centre avant de lancer sur tout Avignon.

In [ ]:
PLACE = "Avignon, Vaucluse, France"
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:2154"          # Lambert 93
VITESSE_MARCHE_M_MIN = 80         # 4,8 km/h, ma vitesse de marche de reference
SEUILS_MIN = [5, 10, 15]          # mes seuils de temps de marche
DIST_MAX_M = 2000                 # distance maximale exploree sur le reseau, en metres
DEV_MODE = True                   # True = petit perimetre central pour tester vite

GTFS_DIR = Path("gtfs_orizo")     # dossier GTFS du reseau Orizo, optionnel
SORTIE = Path("sorties_avignon"); SORTIE.mkdir(exist_ok=True)


## 3. La zone d'étude et les bâtiments

Je récupère le contour d'Avignon, puis mes bâtiments depuis OpenStreetMap. Le bâtiment est mon unité de base, c'est là que vivent les gens, donc c'est là que je veux mesurer l'accessibilité. En mode test, je me limite à un carré de 1,2 kilomètre autour du centre pour que tout aille vite.

In [ ]:
admin = ox.geocode_to_gdf(PLACE)
admin_m = admin.to_crs(CRS_METRIC)

if DEV_MODE:
    centre_m = admin_m.geometry.centroid.iloc[0]
    zone_m = centre_m.buffer(800)   # petit perimetre pour tester vite
    zone = gpd.GeoSeries([zone_m], crs=CRS_METRIC).to_crs(CRS_GEO).iloc[0]
else:
    zone = admin.to_crs(CRS_GEO).geometry.iloc[0]

batiments = tenace(lambda: ox.features_from_polygon(zone, tags={"building": True})).to_crs(CRS_GEO)
batiments = batiments[batiments.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
batiments["geometry"] = batiments.geometry.representative_point()   # un point par batiment
print(len(batiments), "batiments dans ma zone")


Je regarde ma zone et mes bâtiments, pour vérifier que tout est bien chargé avant de continuer.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
admin.to_crs(CRS_GEO).plot(ax=ax1, color="whitesmoke", edgecolor="grey")
gpd.GeoSeries([zone], crs=CRS_GEO).boundary.plot(ax=ax1, color="crimson", linewidth=2)
ax1.set_title("Avignon et ma zone de travail"); ax1.axis("off")
batiments.plot(ax=ax2, color="steelblue", markersize=2)
ax2.set_title("Les batiments"); ax2.axis("off")
plt.tight_layout(); plt.show()


## 4. Les arrêts du réseau Orizo

Je lis les arrêts de bus depuis le GTFS d'Orizo, le réseau d'Avignon. J'avais déjà utilisé ce GTFS pour mon mémoire, donc je sais le manipuler. Si le dossier n'est pas là, je passe cette étape sans bloquer le reste, et je pourrai pointer GTFS_DIR vers mes fichiers plus tard. On télécharge le GTFS d'Orizo sur transport.data.gouv.fr.

In [ ]:
arrets = None
if (GTFS_DIR / "stops.txt").exists():
    stops = pd.read_csv(GTFS_DIR / "stops.txt")
    arrets = gpd.GeoDataFrame(
        stops,
        geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat),
        crs=CRS_GEO,
    )
    arrets = gpd.clip(arrets, gpd.GeoSeries([zone], crs=CRS_GEO))
    fig, ax = plt.subplots(figsize=(8, 8))
    gpd.GeoSeries([zone], crs=CRS_GEO).boundary.plot(ax=ax, color="grey")
    arrets.plot(ax=ax, color="darkorange", markersize=10)
    ax.set_title("Les arrets du reseau Orizo dans ma zone"); ax.axis("off"); plt.show()
    print(len(arrets), "arrets")
else:
    print("Je n'ai pas trouve le GTFS d'Orizo, je passe cette etape pour l'instant.")


## 5. Le réseau piéton avec pandana

Je construis mon réseau de marche à partir du graphe piéton d'OpenStreetMap, puis je le passe à pandana. pandana est très rapide pour calculer, depuis chaque intersection, la distance à l'aménité la plus proche. Le precompute prépare ces calculs jusqu'à ma distance maximale.

In [ ]:
G = tenace(lambda: ox.graph_from_polygon(zone, network_type="walk"))
noeuds, troncons = ox.graph_to_gdfs(G)
troncons = troncons.reset_index()

reseau = pandana.Network(
    noeuds["x"], noeuds["y"],
    troncons["u"], troncons["v"],
    troncons[["length"]],
)
reseau.precompute(DIST_MAX_M)
print("Reseau pieton pret,", len(noeuds), "intersections")


## 6. Le temps de marche jusqu'aux aménités

Je choisis les aménités du quotidien, celles qui comptent pour bien vivre un quartier, la pharmacie, l'école, le commerce alimentaire, la santé et le parc. Pour chacune, je récupère les points depuis OpenStreetMap, je les accroche au réseau, et je demande à pandana la distance jusqu'à la plus proche depuis chaque intersection. Je convertis ensuite les mètres en minutes de marche avec ma vitesse de référence.

In [ ]:
ameniteS = {
    "pharmacie": {"amenity": "pharmacy"},
    "ecole":     {"amenity": "school"},
    "commerce":  {"shop": ["supermarket", "convenience", "bakery"]},
    "sante":     {"amenity": ["hospital", "doctors", "clinic"]},
    "parc":      {"leisure": "park"},
}

acces = pd.DataFrame(index=reseau.node_ids)
for nom, tags in ameniteS.items():
    try:
        g = tenace(lambda tags=tags: ox.features_from_polygon(zone, tags=tags)).to_crs(CRS_GEO)
        pts = g.geometry.representative_point()
        reseau.set_pois(nom, DIST_MAX_M, 1, pts.x, pts.y)
        dist_m = reseau.nearest_pois(DIST_MAX_M, nom, num_pois=1)[1]
        acces["min_" + nom] = dist_m / VITESSE_MARCHE_M_MIN
        print(nom, ",", len(pts), "points")
    except Exception as e:
        print("Pas de donnees pour", nom, ":", e)
acces.head()


## 7. L'accessibilité à l'échelle du bâtiment

Je rattache chaque bâtiment à l'intersection la plus proche, puis je lui donne les temps de marche calculés pour cette intersection. Ainsi chaque bâtiment porte son propre score, par exemple le nombre de minutes pour atteindre la pharmacie la plus proche.

In [ ]:
batiments["node"] = reseau.get_node_ids(batiments.geometry.x, batiments.geometry.y)
for col in acces.columns:
    batiments[col] = batiments["node"].map(acces[col])
batiments[[c for c in batiments.columns if c.startswith("min_")]].describe()


Je cartographie le résultat. Je colore les bâtiments par le temps de marche jusqu'à la pharmacie la plus proche. Le vert est proche, le rouge est loin. On lit tout de suite les quartiers bien servis et ceux qui le sont moins.

In [ ]:
col = "min_pharmacie"
if col in batiments.columns:
    fig, ax = plt.subplots(figsize=(9, 9))
    batiments.plot(ax=ax, column=col, cmap="RdYlGn_r", markersize=6,
                   legend=True, vmin=0, vmax=15,
                   legend_kwds={"label": "minutes de marche jusqu'a la pharmacie"})
    gpd.GeoSeries([zone], crs=CRS_GEO).boundary.plot(ax=ax, color="grey", linewidth=0.8)
    ax.set_title("Avignon, acces a la pharmacie a pied"); ax.axis("off"); plt.show()


## 8. Les bandes d'atteignabilité depuis le centre

Pour finir, je montre jusqu'où on marche depuis le centre d'Avignon en 5, 10 et 15 minutes. Je calcule le temps de marche sur chaque rue, puis je garde les rues atteignables dans chaque seuil. Les bandes suivent le dessin des rues, elles ne sont pas de simples cercles, et s'étirent le long des grands axes.

In [ ]:
for u, v, k, data in G.edges(keys=True, data=True):
    data["minutes"] = data["length"] / VITESSE_MARCHE_M_MIN

centre_node = ox.distance.nearest_nodes(G, zone.centroid.x, zone.centroid.y)

fig, ax = plt.subplots(figsize=(9, 9))
ox.plot_graph(G, ax=ax, node_size=0, edge_linewidth=0.4,
              edge_color="lightgrey", show=False, close=False)
couleurs = {5: "#1a9850", 10: "#fee08b", 15: "#d73027"}
for seuil in sorted(SEUILS_MIN, reverse=True):
    sub = nx.ego_graph(G, centre_node, radius=seuil, distance="minutes")
    e = ox.graph_to_gdfs(sub, nodes=False)
    e.plot(ax=ax, color=couleurs[seuil], linewidth=1.4, label=f"{seuil} min")
ax.legend(title="marche depuis le centre"); ax.set_title("Avignon, atteignabilite a pied depuis le centre")
ax.axis("off"); plt.show()


## 9. Export et conclusion

J'exporte mes bâtiments avec leurs scores, pour pouvoir les réutiliser dans QGIS ou dans une autre analyse.

In [ ]:
cols = ["geometry"] + [c for c in batiments.columns if c.startswith("min_")]
batiments[cols].to_file(SORTIE / "acces_batiments_avignon.gpkg", driver="GPKG")
print("Exporte dans", SORTIE / "acces_batiments_avignon.gpkg")


### Ce que je retiens

J'ai mesuré l'accessibilité à pied dans Avignon, bâtiment par bâtiment, pour les services du quotidien. Mes choix, la vitesse de marche de 4,8 kilomètres par heure, la projection Lambert 93 pour des distances justes, et des seuils à 5, 10 et 15 minutes, viennent d'une logique simple, coller à la marche réelle d'un habitant.

Les limites, je les assume. Mes données viennent d'OpenStreetMap, donc elles valent ce que vaut la saisie locale, une aménité oubliée ne sera pas comptée. Et je mesure la marche sur le réseau, pas le ressenti, une rue sans trottoir compte comme marchable.

Cette analyse s'appuie sur une méthode classique d'accessibilité réseau, que j'ai mise en œuvre sur Avignon avec mon propre code et mes propres choix, pour la maîtriser et la réutiliser sur n'importe quel territoire.